# Business Understanding and Analytic Approach

## 1. Giới thiệu bài toán

Cây xanh đường phố được quản lý như một hạng mục hạ tầng đô thị: cắt tỉa, xử lý cây gãy đổ sau bão, loại bỏ cây nguy hiểm đều do đội bảo trì của thành phố lên lịch. San Francisco công khai cả nhật ký yêu cầu bảo trì (311 Cases) lẫn danh mục cây đường phố (Street Tree List), nhưng phần lớn việc điều phối đội bảo trì vẫn dựa vào kinh nghiệm và tồn đọng công việc, chứ chưa dựa trên dự báo nhu cầu theo khu vực và thời gian — khiến các tuần có bão dễ bị quá tải so với năng lực đội ngũ bình thường.

Dự án này khai thác dữ liệu mở của San Francisco 311 Cases, Street Tree List, và SF Street-Use Permits (dùng làm nguồn phụ ngoại sinh chính) — cùng dữ liệu thời tiết NOAA (dùng mở rộng) — nhằm:

- **Định vị bức tranh nhu cầu**: xác định khu phố, mùa và điều kiện nào có nhu cầu bảo trì cây cao nhất, cao hơn bao nhiêu lần so với khu phố thấp nhất.
- **Định lượng tác động của hoạt động đô thị**: đánh giá hoạt động thi công đường phố (số giấy phép đang hoạt động) và điều kiện thời tiết ảnh hưởng ra sao đến tần suất yêu cầu bảo trì, bằng kiểm định thống kê chứ không chỉ quan sát định tính.
- **Xây dựng nền tảng dự báo đáng tin cậy**: so sánh nhiều họ mô hình — từ mốc đơn giản, hồi quy tuyến tính, cây tăng cường, đến mô hình nền chuỗi thời gian zero-shot — để dự báo nhu cầu 7 ngày tới cho từng khu phố, đồng thời đánh giá độ tin cậy, khả năng diễn giải và khả năng thích ứng của dự báo khi điều kiện vận hành thay đổi.



## 2. Mục tiêu phân tích

Dự án tập trung trả lời 5 câu hỏi nghiên cứu nhằm dự báo nhu cầu bảo trì cây xanh tại San Francisco:
- RQ1 (Cấu trúc nhu cầu): Nhu cầu bảo trì cây xanh biến động thế nào theo khu phố, ngày trong tuần và tháng trong năm?

- RQ2 (So sánh mô hình): Trong sáu họ mô hình dự báo, mô hình nào cho kết quả chính xác nhất cho tuần tới và tổng quát hoá tốt sang khu phố mới?

- RQ3 (Độ tin cậy vận hành): Khoảng tin cậy dự báo và cảnh báo tuần nguy cơ cao đáng tin đến mức nào, và giấy phép thi công đường phố đóng góp bao nhiêu vào độ chính xác?

- RQ4 (Diễn giải mô hình): Vì sao hoạt động thi công đường phố được xếp hạng quan trọng nhưng loại bỏ khỏi mô hình lại không làm giảm độ chính xác?

- RQ5 (Thích ứng với thay đổi): Tái huấn luyện định kỳ có giúp mô hình thích ứng tốt hơn khi điều kiện vận hành thay đổi đột ngột không?

## 3. Các yếu tố ảnh hưởng đến nhu cầu bảo trì cây xanh

Dựa trên ba nguồn dữ liệu chính (311 Cases, Street Tree List, Street-Use Permits) và nguồn mở rộng (thời tiết NOAA), các yếu tố được đưa vào phân tích gồm:

- Thời gian (requested_datetime): quyết định tính chu kỳ của yêu cầu bảo trì (ngày trong tuần, mùa vụ).

- Khu phố (nbhd): đặc trưng không gian, mật độ dân cư và hạ tầng khác nhau giữa 41 khu phố.

- Hoạt động thi công đường phố (permits_active): số giấy phép thi công đang hoạt động, có thể làm tăng va chạm vào cây.

- Điều kiện thời tiết (PRCP, AWND, TMAX): mưa lớn và gió mạnh làm tăng nguy cơ cây gãy, đổ cành.

- Đặc điểm cây (dbh, ngày trồng, loài): phản ánh tuổi và tình trạng quần thể cây tại từng khu phố.
## 4. Cách tiếp cận phân tích

Để trả lời năm câu hỏi nghiên cứu trên, dự án thực hiện theo pipeline sáu bước:

1. **Thu thập và làm sạch dữ liệu:**

- Tải dữ liệu 311 Cases, Street Tree List, Street-Use Permits và thời tiết NOAA qua API công khai của thành phố.

- Xử lý dữ liệu thiếu, trùng lặp và sai định dạng thời gian.

- Ghép các nguồn dữ liệu theo khu phố và thời gian, không để rò rỉ thông tin tương lai.

2. **Phân tích SQL (DuckDB) — trả lời RQ1:**

- Tổng hợp nhu cầu bảo trì theo khu phố, ngày trong tuần và tháng.

- Tạo đặc trưng trễ và trung bình trượt phục vụ dự báo.

- Kiểm định thống kê để so sánh khác biệt giữa các khu phố và các yếu tố liên quan.

3. **Phân tích khám phá dữ liệu (EDA) bằng Python — trả lời RQ1:**

- Đo mức chênh lệch nhu cầu giữa các khu phố và tính chu kỳ theo lịch.

- Đánh giá tương quan giữa nhu cầu với thời tiết và hoạt động thi công đường phố.

4. **Trực quan hoá dữ liệu:**

- Vẽ biểu đồ phân phối và chuỗi thời gian nhu cầu theo khu phố.

- Vẽ biểu đồ mùa vụ theo giờ, ngày, tháng để làm nổi bật quy luật chu kỳ.

5. **Xây dựng và so sánh mô hình dự báo — trả lời RQ2:**

- Chia dữ liệu theo thời gian để huấn luyện và kiểm thử, tránh rò rỉ tương lai.

- Huấn luyện và so sánh sáu mô hình dự báo bằng nhiều chỉ số đánh giá.

- Kiểm tra khả năng tổng quát hoá sang khu phố mới và theo nhiều mốc thời gian.

6. **Đánh giá độ tin cậy, diễn giải và khả năng thích ứng — trả lời RQ3, RQ4, RQ5:**

- Tính khoảng tin cậy dự báo và xây dựng cơ chế cảnh báo tuần nguy cơ cao.

- Dùng SHAP để xác định mức độ quan trọng của từng yếu tố đầu vào.

- So sánh mô hình cố định với mô hình tái huấn luyện định kỳ để đánh giá khả năng thích ứng.
## 5. Kỳ vọng kết quả

- Tri thức định lượng: Làm rõ mức độ chênh lệch nhu cầu bảo trì cây xanh giữa các khu phố San Francisco, cùng mức độ ảnh hưởng thực sự của hoạt động thi công đường phố và thời tiết đến nhu cầu đó.

- Giải pháp hỗ trợ quyết định: Cung cấp mô hình dự báo đáng tin cậy, có khoảng tin cậy và cảnh báo sớm, hỗ trợ điều phối đội bảo trì cây xanh chủ động thay vì phản ứng bị động.

- Sản phẩm học thuật: Đóng gói hoàn chỉnh bộ mã nguồn Jupyter Notebook có thể tái tạo, nhật ký sử dụng AI (Audit Log), cùng bản thảo nghiên cứu theo mẫu hội nghị (Springer LNCS) đáp ứng chuẩn mực học thuật quốc tế.
